# 에이전트 계획 수립(Agent Planning)

이 노트북은 agent가 복잡한 요청을 바로 답하지 않고, 먼저 실행 계획(plan)으로 바꾸는 이유를 다룬다. 특히 ReAct, planner-executor, decomposition 세 전략이 각각 어떤 질문에 잘 맞는지 비교하고, 계획이 실제 executor와 연결될 때 어떤 장점이 생기는지 실험해본다.

## 학습 목표
- ReAct, planner-executor, decompose 세 planning 전략의 차이를 설명할 수 있다.
- 어떤 질문이 어떤 planning 스타일에 더 잘 맞는지 감을 잡는다.
- `PlanExecutor`가 handler를 등록하고 step별로 실행하는 구조를 이해한다.
- plan이 단순한 텍스트 목록이 아니라, 실행 제어와 디버깅을 위한 인터페이스라는 점을 이해한다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


## 개념 설명

planning notebook도 먼저 실행 환경을 확인한다. planning 자체는 추상적 개념처럼 보이지만, 이 노트북은 `src/planner_extended.py`의 클래스를 직접 불러와 실행하므로 모듈 경로와 커널 상태가 안정적이어야 한다.

- **목적**: 현재 커널이 올바른 Python 환경을 사용하고 있는지 확인한다.
- **핵심 로직**: 프로젝트 루트를 찾기 전에 `sys.executable`을 출력해 환경을 점검한다.
- **주요 파라미터/변수**:
  - `sys.executable`: 현재 노트북이 연결된 Python 실행 파일이다.

짧은 셀이지만, 이후 `PlanGenerator`와 `PlanExecutor`가 같은 환경에서 돌아가도록 보장하는 시작점이다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

이 setup 셀은 planner 관련 클래스를 불러오고, 노트북이 어느 프로젝트 루트를 기준으로 움직이는지 다시 한 번 맞춘다. 기존 `src/planner.py`는 간단한 기본 planner를 유지하고, 여기서는 교육용 확장 모듈인 `planner_extended.py`를 사용해 전략 차이를 더 분명히 보여준다.

- **목적**: plan 생성기와 실행기를 실습 가능한 상태로 준비한다.
- **핵심 로직**: `PROJECT_ROOT`를 조정하고 `PlanGenerator`, `PlanExecutor`를 import한 뒤, `generator = PlanGenerator()`로 생성기 인스턴스를 만든다.
- **주요 파라미터/변수**:
  - `PROJECT_ROOT`: planner 모듈을 찾기 위한 기준 경로이다.
  - `generator`: 여러 planning 전략을 비교 생성하는 객체이다.

setup 셀을 따로 두는 이유는, 이후 셀에서 전략 비교와 실행기를 분리해 설명하기 위함이다. planning을 배우려면 생성과 실행을 한꺼번에 보지 않고 나눠서 보는 편이 더 이해하기 쉽다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.planner_extended import PlanExecutor, PlanGenerator

pd.set_option('display.max_colwidth', 140)
generator = PlanGenerator()


## 계획 전략 비교

이 셀은 같은 작업을 세 가지 전략으로 계획해본다.
- **ReAct**: 관찰(observe)과 사고(think), 행동(act)를 반복하는 스타일에 가깝다. 모호하고 탐색적인 질문에 잘 맞는다.
- **planner_executor**: 먼저 비교적 안정적인 계획을 세우고, 그 계획을 별도의 실행기가 처리하는 구조다. 안전성과 추적성이 중요할 때 유리하다.
- **decompose**: 넓은 작업을 작은 하위 작업으로 기계적으로 쪼개는 데 강하다. 복합 과업을 명시적으로 분해할 때 좋다.

- **목적**: 하나의 작업을 세 전략으로 나누었을 때 step 수와 구조가 어떻게 달라지는지 본다.
- **핵심 로직**: `generator.compare_strategies(planning_task)`가 동일한 task에 대해 세 전략의 plan 리스트를 한 번에 생성한다.
- **주요 파라미터/변수**:
  - `planning_task`: 여러 단계를 요구하는 대표 과업이다.
  - `strategy_comparison`: 전략별 plan 결과 딕셔너리이다.

예를 들어 아래 코드에서:
- `Find the rollout date, compare it to the pilot window, and summarize why the timing matters.`: retrieval, comparison, synthesis가 모두 필요한 복합 질문이다.
- `{strategy: len(plan) for strategy, plan in strategy_comparison.items()}`: 전략마다 얼마나 세분화했는지 빠르게 보여준다.

어떤 질문에 어떤 전략이 적합한가? exploratory QA는 ReAct, 예측 가능한 업무 자동화는 planner-executor, 범위가 넓은 리서치 작업은 decomposition이 잘 맞는 편이다.

### 실제 구현 펼쳐보기: `PlanStep`와 `PlanGenerator.generate_plan()`
```python
class PlanStep:
    step_id: str
    objective: str
    rationale: str
    tool_hint: str | None = None

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)
```

```python
    def generate_plan(self, task: str, strategy: str = "planner_executor") -> list[dict[str, Any]]:
        normalized_task = normalize_text(task)
        strategy_name = strategy if strategy in self.STRATEGIES else "planner_executor"

        if strategy_name == "react":
            steps = [
                PlanStep("react_1", f"Observe the task: {normalized_task}", "Start by reading the request carefully."),
                PlanStep("react_2", "Think about what information is missing.", "Identify gaps before acting."),
                PlanStep("react_3", "Act with the best available tool or retrieval step.", "Collect evidence or perform a calculation.", "tool_or_retrieval"),
                PlanStep("react_4", "Observe the result and check whether it is enough.", "Use tool output to decide the next move."),
                PlanStep("react_5", "Respond with the grounded result.", "Finish only after the evidence supports the answer."),
            ]
            return [step.to_dict() for step in steps]

        if strategy_name == "decompose":
            clauses = self._decompose_task(normalized_task)
            steps = [
                PlanStep(
                    step_id=f"decompose_{index}",
                    objective=clause,
                    rationale="Solve one sub-problem at a time to reduce cognitive load.",
                    tool_hint="retrieval" if "find" in clause.lower() or "lookup" in clause.lower() else None,
                )
                for index, clause in enumerate(clauses, start=1)
            ]
            return [step.to_dict() for step in steps]

        steps = [
            PlanStep("plan_1", f"Clarify the objective: {normalized_task}", "Define what a successful answer should contain."),
            PlanStep("plan_2", "Gather evidence for each important part of the task.", "Planner-executor systems work best with explicit evidence collection.", "retrieval"),
            PlanStep("plan_3", "Execute any focused tools needed for calculations or lookups.", "Use tools only after the sub-tasks are clear.", "tool"),
            PlanStep("plan_4", "Synthesize the final response from evidence and tool outputs.", "Combine the executor outputs into a coherent answer."),
        ]
        return [step.to_dict() for step in steps]
```

전략별 적합도:
- `react`: 탐색적 질문, 중간에 방향이 바뀔 수 있는 상황
- `planner_executor`: 목표가 비교적 명확하고 단계적으로 풀 수 있는 작업
- `decompose`: 복합 요청을 하위 문제로 쪼갤 수 있는 경우


In [ ]:
planning_task = 'Find the rollout date, compare it to the pilot window, and summarize why the timing matters.'
strategy_comparison = generator.compare_strategies(planning_task)
{
    strategy: len(plan)
    for strategy, plan in strategy_comparison.items()
}


## 계획 구조 읽기

step 수만으로는 계획의 질을 판단할 수 없다. 실제 step이 어떤 형태로 구성됐는지 봐야 한다. 이 셀은 전략별 plan을 데이터프레임으로 바꿔, 각 단계의 objective와 타입을 눈으로 비교하게 해준다.

- **목적**: 세 planning 전략의 step 구조 차이를 표 수준에서 읽는다.
- **핵심 로직**: 전략별 plan 리스트를 `pd.DataFrame`으로 바꾸어 나란히 확인한다.
- **주요 파라미터/변수**:
  - `plan_frames['react']`: 사고-행동 루프 중심의 단계 구조를 보여준다.
  - `plan_frames['planner_executor']`: 실행 친화적인 단계 구성을 보여준다.
  - `plan_frames['decompose']`: 작업 분해 중심 구조를 보여준다.

결과를 볼 때는 step 개수보다 `objective`의 명확성을 보자. 너무 큰 step은 executor가 해석하기 어렵고, 너무 잘게 쪼개진 step은 오버헤드가 커질 수 있다.

### 실제 구현 펼쳐보기: `compare_strategies()`
```python
    def compare_strategies(self, task: str) -> dict[str, list[dict[str, Any]]]:
        return {strategy: self.generate_plan(task, strategy=strategy) for strategy in self.STRATEGIES}
```

이 함수는 같은 task에 대해 세 전략을 한 번에 돌려 비교용 출력으로 바꾼다. 교육용 노트북에서 특히 유용한 이유는, 전략 간 차이를 동일 입력 기준으로 바로 눈으로 볼 수 있기 때문이다.


In [ ]:
plan_frames = {
    strategy: pd.DataFrame(plan)
    for strategy, plan in strategy_comparison.items()
}
plan_frames['react'], plan_frames['planner_executor'], plan_frames['decompose']


## PlanExecutor 이해하기

planning이 실제로 유용해지려면 executor와 연결되어야 한다. `PlanExecutor`는 각 step type에 맞는 handler를 등록해, 계획을 실제 실행 로그로 바꾸는 구조다. 이 패턴의 장점은 planner와 executor를 느슨하게 결합할 수 있다는 점이다. planner는 "무엇을 해야 하는가"를 만들고, executor는 "어떻게 실행할 것인가"를 담당한다.

- **목적**: 계획 단계와 실행 단계를 연결하는 executor 패턴을 이해한다.
- **핵심 로직**: `register_handler(step_type, handler)`로 처리 함수를 등록한 뒤, `execute(plan)`이 step을 순서대로 돌며 실행 로그를 만든다.
- **주요 파라미터/변수**:
  - `executor`: handler 기반 실행기 객체이다.
  - `register_handler('retrieval', ...)`: retrieval 타입 step을 만났을 때 어떤 동작을 할지 연결한다.
  - `execution_log`: 각 단계가 어떤 상태로 끝났는지 보여주는 실행 결과 표이다.

이 구조는 실무에서 매우 유용하다. 새 tool이나 새 step type을 추가할 때 planner 전체를 고치지 않고 handler만 등록하면 되기 때문이다.

### 실제 구현 펼쳐보기: `PlanExecutor`
```python
class PlanExecutor:
    def __init__(self) -> None:
        self.handlers: dict[str, Handler] = {}

    def register_handler(self, name: str, handler: Handler) -> None:
        self.handlers[name] = handler

    def execute(self, plan: list[dict[str, Any]]) -> list[dict[str, Any]]:
        execution_log: list[dict[str, Any]] = []
        for step in plan:
            tool_hint = step.get("tool_hint")
            handler = self.handlers.get(tool_hint or "", self._default_handler)
            result = handler(step)
            execution_log.append(
                {
                    "step_id": step["step_id"],
                    "objective": step["objective"],
                    "tool_hint": tool_hint,
                    "status": result.get("status", "completed"),
                    "output": result.get("output", ""),
                }
            )
        return execution_log

    @staticmethod
    def _default_handler(step: dict[str, Any]) -> dict[str, Any]:
        return {
            "status": "completed",
            "output": f"Executed step: {step['objective']}",
        }
```

handler 등록 패턴이 좋은 이유:
- planner는 계획만 만들고, executor는 step type에 맞는 handler를 찾아 실행한다.
- 새 도구나 기능을 추가할 때 `PlanExecutor` 본문을 크게 뜯지 않아도 handler만 등록하면 된다.
- 즉, 확장성은 높이고 결합도는 낮춘다.


In [ ]:
executor = PlanExecutor()
executor.register_handler('retrieval', lambda step: {'status': 'completed', 'output': f"Retrieved evidence for: {step['objective']}"})
executor.register_handler('tool', lambda step: {'status': 'completed', 'output': f"Ran a focused tool for: {step['objective']}"})
planner_executor_plan = generator.generate_plan(planning_task, strategy='planner_executor')
execution_log = executor.execute(planner_executor_plan)
pd.DataFrame(execution_log)


## 실험

이 셀은 서로 다른 세 과업을 `decompose` 전략으로 계획해보고, 첫 단계가 어떻게 달라지는지 비교한다. 같은 planner라도 입력 task가 바뀌면 분해 방식이 달라져야 자연스럽다.

- **목적**: 과업의 성격에 따라 생성되는 plan 길이와 첫 단계가 어떻게 달라지는지 확인한다.
- **핵심 로직**: `experiment_tasks`를 순회하며 `generate_plan(task, strategy='decompose')`를 호출하고, 각 task의 plan length와 first step을 표로 저장한다.
- **주요 파라미터/변수**:
  - `experiment_tasks`: 요약, 날짜+설명, 검색+계산+작성 등 난이도가 다른 과업 목록이다.
  - `plan_length`: 과업 분해 정도를 보여주는 길이 지표이다.
  - `first_step`: planner가 해당 과업에서 무엇을 가장 먼저 해야 한다고 판단했는지 보여준다.

이 결과를 보면 planner가 과업의 형태를 어느 정도 읽고 있는지 감을 잡을 수 있다. 좋은 planner는 항상 같은 첫 단계를 내놓지 않고, task 성격에 맞춰 우선순위를 조정한다.

### 실제 구현 펼쳐보기: `decompose` 전략의 내부 분해
```python
    def _decompose_task(self, task: str) -> list[str]:
        separators = (" and ", " then ", ",")
        parts = [task]
        for separator in separators:
            if separator in task.lower():
                parts = [normalize_text(part) for part in task.split(separator) if normalize_text(part)]
                break
        if len(parts) == 1:
            return [
                f"Identify the core request in: {task}",
                "Collect the evidence needed for the request.",
                "Produce a grounded final answer.",
            ]
        return [f"Handle sub-task: {part}" for part in parts]
```

핵심 해설:
- `and`, `then`, `,` 같은 separator를 보고 하위 절을 나눈다.
- 분리되지 않으면 기본 3단계 템플릿으로 fallback한다.
- 단순하지만, 복합 질문을 작은 unit으로 나누는 사고방식을 잘 보여준다.


In [ ]:
experiment_tasks = [
    'Summarize the rollout plan.',
    'Find the rollout date and explain who needs to know it.',
    'Search the policy goals, calculate the pilot duration, and draft a short update.',
]
experiment_rows = []
for task in experiment_tasks:
    plan = generator.generate_plan(task, strategy='decompose')
    experiment_rows.append({'task': task, 'plan_length': len(plan), 'first_step': plan[0]['objective']})
pd.DataFrame(experiment_rows)


## 결과 해석

이 마지막 분석 셀은 세 전략의 강점을 한 줄씩 요약한다. 계획 전략은 정답이 하나인 문제가 아니므로, 각 전략이 어떤 상황에서 더 적합한지 맥락과 함께 읽는 것이 중요하다.

- **목적**: 전략 선택 기준을 짧은 표로 정리한다.
- **핵심 로직**: `analysis_frame`에 전략별 강점을 요약해 저장한다.
- **주요 파라미터/변수**:
  - `strategy`: 비교 대상 planning 방식이다.
  - `strength`: 그 전략이 특히 잘 맞는 상황에 대한 한 줄 설명이다.

이 표를 읽을 때는 "어느 전략이 최고인가"보다 "어느 질문에 어떤 전략을 쓰면 좋은가"를 중심으로 보자. agent 설계에서 planning은 모델 성능이 아니라 제어 구조 선택의 문제이기도 하다.

### 전략 비교 해석 가이드
- ReAct는 `observe -> think -> act -> observe -> respond`처럼 상호작용 루프를 강조한다.
- planner-executor는 `clarify -> gather -> execute -> synthesize`처럼 안정적인 파이프라인에 가깝다.
- decompose는 복합 질의를 잘게 나눠 cognitive load를 낮춘다.

💡 면접 포인트: ReAct는 탐색 루프, Plan-and-Execute는 명시적 계획과 실행 분리를 강조하는 패턴이라는 차이를 말할 수 있어야 한다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'strategy': 'react', 'strength': 'good for iterative observe-think-act loops'},
        {'strategy': 'planner_executor', 'strength': 'good for explicit planning and safe execution'},
        {'strategy': 'decompose', 'strength': 'good for turning broad tasks into manageable chunks'},
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 planning은 단순한 설명 문장이 아니라, 실행 흐름을 구조화하는 제어 장치라는 점을 확인했다. ReAct는 탐색적 루프에, planner-executor는 명시적 실행 제어에, decomposition은 복합 작업 분해에 강점을 가진다.

또한 `PlanExecutor`처럼 handler 기반 실행기를 붙이면, 계획이 실제 동작 가능한 인터페이스로 바뀐다. 이때 planner와 executor를 분리해두면 기능을 확장해도 구조가 덜 흔들린다.

💡 면접 포인트: "planning은 모델이 더 똑똑해 보이게 하는 장식이 아니라, 복잡한 요청을 추적 가능하고 실행 가능한 단계로 바꾸는 시스템 설계 요소"라고 설명하면 좋다.

계획 전략의 정답은 하나가 아니다. 중요한 것은 질문 유형에 맞는 제어 구조를 선택하는 것이다. 즉, planning은 성능 향상 장치이면서 동시에 시스템 설명 가능성을 높이는 인터페이스이기도 하다.
